In [ ]:
from trainer import BaseTrainModule, Trainer
from model import SvdHClassification
from dataset_pre import Data_Loader

import torch
from torchvision import transforms
import torch.nn.functional as F
from torch.utils.data import Dataset, random_split
import matplotlib.pyplot as plt
from torch import optim
import torch.nn as nn
import os

ROOT_PATH = os.path.abspath(os.path.join(os.path.join(os.path.join(os.getcwd(), os.pardir), os.pardir), os.pardir))

class MyTrainModule(BaseTrainModule):
    def __init__(self, reduce):
        super().__init__()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        self.reduce = reduce

        self.model = SvdHClassification(in_channels=1, out_channels=5)
        self.model = nn.DataParallel(self.model)
        self.model.to(device=self.device)

    def configure_lossfunctions(self):
        self.criterion = nn.CrossEntropyLoss()

    def configure_optimizers(self, lr):
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr, betas=(0.5, 0.999), weight_decay=1e-3)
        # self.optimizer = optim.SGD(self.model.parameters(), lr=lr)

        return self.optimizer

    def configure_scheduler(self, optimizers):
        optimizer = optimizers
        self.scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer, step_size=300, gamma=0.5)

        return (self.scheduler)

    def configure_logs(self):
        # learning rate
        self.set_log(name='Lr', obj=self.optimizer, category='lr')
        # loss function
        self.set_log(name='Loss_jsw', obj=self.criterion, category='loss', mode='train')
        self.set_log(name='Loss_jsw_val', obj=self.criterion, category='loss', mode='valid')

    def training_step(self, batch_idx, batch):
        self.model.train()
        image, label = batch

        self.optimizer.zero_grad()
        pre_svdh = self.model(image)
        print('{} Pre: {}, GT: {}'.format(self.reduce, F.softmax(pre_svdh, dim=1).cpu().detach().numpy()[0], label.cpu().detach().numpy()[0]))
        
        loss = self.criterion(pre_svdh, label)

        loss.backward()
        self.optimizer.step()

        return loss

    def validation_step(self, batch_idx, batch):
        self.model.eval()
        image, label = batch

        pre_svdh = self.model(image)
        
        loss_val = self.criterion(pre_svdh, label)
        print('\tPre: {}, GT: {}'.format(F.softmax(pre_svdh, dim=1).cpu().detach().numpy()[0], label.cpu().detach().numpy()[0]))
        
        image_list = [image]

        return loss_val, image_list

    def configure_saveprocess(self):
        self.set_save_parameter(model=self.model, loss_name='Loss_jsw',
                                save_path=ROOT_PATH + '/experiments/Exp_downstream/SvdH_evaluation/parameter/best_svdH_model_pre_{}.pth'.format(self.reduce))

    def show_single_log_image(self, image_box):
        image = image_box
        plt.imshow(image[0][0])
        plt.show()


if __name__ == "__main__":
    # Data Loading
    image_size = 256
    transform = transforms.Compose([transforms.Resize((image_size, image_size)),
                                    transforms.ToTensor(),
                                    transforms.Normalize(0, 1),
                                    transforms.RandomRotation(45)
                                    ])
    
    reduce_list = [20]

    for REDUCE in reduce_list:
        print(REDUCE)

        dataset = Data_Loader(ROOT_PATH + '/experiments/Exp_downstream/SvdH_evaluation/pre_SvdH_data_{}.json'.format(REDUCE), transform)
        train_size = int(0.8 * len(dataset)) 
        test_size = len(dataset) - train_size  
        train_dataset, valid_dataset = random_split(dataset, [train_size, test_size])

        mytrainmodule = MyTrainModule(reduce=REDUCE)
        trainer = Trainer(train_module=mytrainmodule, train_dataset=train_dataset, valid_dataset=valid_dataset)
        trainer.configure(batch_size=24, epochs=150, device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
                        result_number=1, lr=1e-5)
        trainer.fit()
